In [ ]:
"""
Fine-Tuning Ciblé pour ConvNext MSF
===================================
Améliore spécifiquement tan_spot et leaf_blight
en ajustant seulement le classifier head
"""

import os
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Ajouter le chemin du script principal pour importer les fonctions
sys.path.append(os.path.dirname(os.path.abspath(__file__)))

# Importer les fonctions du script principal
# Note: Si vous utilisez un notebook, vous pouvez copier les fonctions nécessaires
# ou les importer depuis train_convnext.ipynb si configuré

# =============================================================================
# Configuration Fine-Tuning
# =============================================================================
SAVE_DIR = '../../saved_models_and_data'
SPLIT_OUTPUT_DIR = '../../dataset_split'
IMAGE_SIZE = (320, 320)
BATCH_SIZE = 24

# Modèle pré-entraîné à charger
PRETRAINED_MODEL_PATH = os.path.join(SAVE_DIR, 'wheat_disease_convnext_model.pth')
LEGACY_MODEL_PATH = os.path.join(SAVE_DIR, 'best_model_simple.pth')

# Paramètres Fine-Tuning
FINE_TUNE_EPOCHS = 10
FINE_TUNE_LR = 1e-5  # Learning rate très faible
FINE_TUNE_WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1

# Classes difficiles à améliorer
DIFFICULT_CLASSES = ['tan_spot', 'leaf_blight']

# Fichier de sortie
FINE_TUNED_MODEL_PATH = os.path.join(SAVE_DIR, 'fine_tuned_convnext_msf.pth')

os.makedirs(SAVE_DIR, exist_ok=True)

# =============================================================================
# Importer/Copier les classes et fonctions nécessaires
# =============================================================================
# Note: Copiez ces sections depuis train_convnext.ipynb ou importez-les

from torchvision import transforms, models
from torch.utils.data import Dataset, WeightedRandomSampler
from PIL import Image
import shutil

# Focal Loss (copier depuis train_convnext.ipynb)
class FocalLoss(nn.Module):
    """Improved Focal Loss with adaptive gamma and progressive hard example boost"""
    def __init__(self, gamma=2.5, class_weights=None, device='cpu', 
                 adaptive_gamma=False, difficult_class_indices=None):
        super().__init__()
        self.gamma = gamma
        self.adaptive_gamma = adaptive_gamma
        self.difficult_class_indices = difficult_class_indices or []
        self.class_weights = class_weights
        if class_weights is not None:
            self.class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
    
    def forward(self, inputs, targets, label_smoothing=0.1):
        ce_loss = F.cross_entropy(
            inputs, targets, 
            reduction='none', 
            weight=self.class_weights,
            label_smoothing=label_smoothing
        )
        pt = torch.exp(-ce_loss)
        
        if self.adaptive_gamma and len(self.difficult_class_indices) > 0:
            gamma = self.gamma * torch.ones_like(targets, dtype=torch.float32)
            for idx in self.difficult_class_indices:
                gamma[targets == idx] = self.gamma * 1.3
        else:
            gamma = self.gamma
        
        focal_loss = (1 - pt) ** gamma * ce_loss
        
        probs = F.softmax(inputs, dim=1)
        target_probs = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        
        confidence_mask_low = target_probs < 0.3
        confidence_mask_medium = (target_probs >= 0.3) & (target_probs < 0.6)
        confidence_mask_high = target_probs >= 0.6
        
        hard_example_boost = torch.zeros_like(target_probs)
        hard_example_boost[confidence_mask_low] = 0.3 * (1 - target_probs[confidence_mask_low]) ** 2
        hard_example_boost[confidence_mask_medium] = 0.2 * (1 - target_probs[confidence_mask_medium]) ** 2
        hard_example_boost[confidence_mask_high] = 0.1 * (1 - target_probs[confidence_mask_high]) ** 2
        
        focal_loss = focal_loss * (1 + hard_example_boost)
        
        return focal_loss.mean()

# Multi-Scale Fusion (copier depuis train_convnext.ipynb)
class MultiScaleFusion(nn.Module):
    """Multi-scale feature fusion with 3 branches"""
    def __init__(self, channels):
        super().__init__()
        self.branch1 = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, groups=channels//8),
            nn.Conv2d(channels, channels, 1),
            nn.BatchNorm2d(channels),
            nn.GELU()
        )
        self.branch2 = nn.Sequential(
            nn.Conv2d(channels, channels, 5, padding=2, groups=channels//8),
            nn.Conv2d(channels, channels, 1),
            nn.BatchNorm2d(channels),
            nn.GELU()
        )
        self.branch3 = nn.Sequential(
            nn.Conv2d(channels, channels, 7, padding=3, groups=channels//8),
            nn.Conv2d(channels, channels, 1),
            nn.BatchNorm2d(channels),
            nn.GELU()
        )
        self.fusion = nn.Sequential(
            nn.Conv2d(channels * 3, channels, 1),
            nn.BatchNorm2d(channels)
        )
    
    def forward(self, x):
        f1 = self.branch1(x)
        f2 = self.branch2(x)
        f3 = self.branch3(x)
        concat = torch.cat([f1, f2, f3], dim=1)
        fused = self.fusion(concat)
        return fused

# Dataset (copier depuis train_convnext.ipynb)
class WheatDiseaseDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        self.samples = []
        for cls in self.classes:
            class_dir = os.path.join(root_dir, cls)
            if not os.path.isdir(class_dir):
                continue
            for img_file in os.listdir(class_dir):
                if img_file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    path = os.path.join(class_dir, img_file)
                    self.samples.append((path, self.class_to_idx[cls]))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, target = self.samples[idx]
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, target

# Build Model (copier depuis train_convnext.ipynb)
def build_model(num_classes):
    """Build ConvNeXt with Multi-Scale Fusion"""
    model = models.convnext_base(pretrained=True)
    in_features = model.classifier[2].in_features
    model.fusion = MultiScaleFusion(in_features)
    model.classifier = nn.Sequential(
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.LayerNorm(in_features),
        nn.Dropout(0.2),
        nn.Linear(in_features, 768),
        nn.GELU(),
        nn.Dropout(0.3),
        nn.Linear(768, 384),
        nn.GELU(),
        nn.Dropout(0.2),
        nn.Linear(384, num_classes)
    )
    
    def forward(x):
        x = model.features(x)
        x = model.fusion(x)
        x = model.classifier(x)
        return x
    
    model.forward = forward
    return model

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((int(IMAGE_SIZE[0] * 1.15), int(IMAGE_SIZE[1] * 1.15))),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(35),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def get_dataloaders():
    """Load data splits"""
    split_dirs = [os.path.join(SPLIT_OUTPUT_DIR, s) for s in ['train', 'val', 'test']]
    
    train_dataset = WheatDiseaseDataset(split_dirs[0], train_transform)
    val_dataset = WheatDiseaseDataset(split_dirs[1], test_transform)
    test_dataset = WheatDiseaseDataset(split_dirs[2], test_transform)
    
    # Weighted sampling pour classes difficiles
    targets = [s[1] for s in train_dataset.samples]
    class_counts = np.bincount(targets)
    class_weights = 1.0 / class_counts
    
    class_names = train_dataset.classes
    if 'tan_spot' in class_names:
        tan_idx = class_names.index('tan_spot')
        class_weights[tan_idx] *= 2.5  # Boost pour fine-tuning
    if 'leaf_blight' in class_names:
        leaf_idx = class_names.index('leaf_blight')
        class_weights[leaf_idx] *= 3.0  # Boost pour fine-tuning
    
    class_weights = class_weights / class_weights.sum() * len(class_weights)
    sample_weights = [class_weights[t] for t in targets]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    return train_loader, val_loader, test_loader, train_dataset.classes, class_weights

# =============================================================================
# Fonction Fine-Tuning
# =============================================================================
def fine_tune_difficult_classes(
    model, 
    train_loader, 
    val_loader, 
    device,
    class_names,
    class_weights,
    difficult_classes=['tan_spot', 'leaf_blight'],
    num_epochs=10,
    learning_rate=1e-5
):
    """
    Fine-tuning ciblé pour améliorer tan_spot et leaf_blight
    """
    print("\n" + "="*80)
    print("🔒 FREEZING BACKBONE AND MSF MODULE...")
    print("="*80)
    
    # 1. Freeze le backbone et MSF
    for param in model.features.parameters():
        param.requires_grad = False
    for param in model.fusion.parameters():
        param.requires_grad = False
    
    # 2. Seulement le classifier est entraînable
    trainable_params = sum(p.numel() for p in model.classifier.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"✅ Training only classifier head: {trainable_params:,} / {total_params:,} parameters")
    
    # 3. Optimizer avec LR très faible
    optimizer = optim.AdamW(
        model.classifier.parameters(), 
        lr=learning_rate,
        weight_decay=FINE_TUNE_WEIGHT_DECAY
    )
    
    # 4. Loss avec focus sur classes difficiles
    difficult_class_indices = []
    if 'tan_spot' in class_names:
        difficult_class_indices.append(class_names.index('tan_spot'))
    if 'leaf_blight' in class_names:
        difficult_class_indices.append(class_names.index('leaf_blight'))
    
    criterion = FocalLoss(
        gamma=3.0,  # Plus agressif pour fine-tuning
        class_weights=class_weights,
        device=device,
        adaptive_gamma=False,
        difficult_class_indices=difficult_class_indices
    )
    
    # 5. Entraînement
    print(f"\n🎯 FINE-TUNING FOR: {difficult_classes}")
    print(f"   Learning rate: {learning_rate}")
    print(f"   Epochs: {num_epochs}")
    print(f"   Focus: tan_spot + leaf_blight F1 scores")
    print("="*80 + "\n")
    
    best_difficult_f1 = 0.0
    best_overall_acc = 0.0
    patience_counter = 0
    history = {
        'train_loss': [],
        'val_acc': [],
        'tan_spot_f1': [],
        'leaf_blight_f1': [],
        'difficult_f1': []
    }
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = 0.0
        train_total = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels, label_smoothing=LABEL_SMOOTHING)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            train_total += inputs.size(0)
        
        # Validation
        model.eval()
        val_metrics = evaluate_validation(model, val_loader, device, class_names)
        
        train_loss /= train_total
        difficult_f1 = (val_metrics['tan_spot_f1'] + val_metrics['leaf_blight_f1']) / 2
        
        history['train_loss'].append(train_loss)
        history['val_acc'].append(val_metrics['overall_acc'])
        history['tan_spot_f1'].append(val_metrics['tan_spot_f1'])
        history['leaf_blight_f1'].append(val_metrics['leaf_blight_f1'])
        history['difficult_f1'].append(difficult_f1)
        
        print(f"Epoch {epoch+1:2d}/{num_epochs} - "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Acc: {val_metrics['overall_acc']:.4f} | "
              f"tan_spot F1: {val_metrics['tan_spot_f1']:.4f} | "
              f"leaf_blight F1: {val_metrics['leaf_blight_f1']:.4f} | "
              f"Difficult F1: {difficult_f1:.4f}")
        
        # Sauvegarder si meilleur
        if difficult_f1 > best_difficult_f1:
            best_difficult_f1 = difficult_f1
            best_overall_acc = val_metrics['overall_acc']
            torch.save(model.state_dict(), FINE_TUNED_MODEL_PATH)
            print(f"  ✅ Nouveau meilleur! Difficult F1: {difficult_f1:.4f}")
            patience_counter = 0
        else:
            patience_counter += 1
        
        # Early stopping (optionnel)
        if patience_counter >= 5:
            print(f"\n⏹️  Early stopping (no improvement for 5 epochs)")
            break
    
    print(f"\n✅ Fine-tuning complete!")
    print(f"   Best Difficult F1: {best_difficult_f1:.4f}")
    print(f"   Best Overall Acc: {best_overall_acc:.4f}")
    
    return model, history

# =============================================================================
# Fonction d'évaluation
# =============================================================================
def evaluate_validation(model, val_loader, device, class_names):
    """Évaluer sur validation set"""
    model.eval()
    y_true, y_pred = [], []
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())
    
    report = classification_report(
        y_true, y_pred, 
        target_names=class_names, 
        output_dict=True,
        zero_division=0
    )
    
    metrics = {
        'overall_acc': report['accuracy'],
        'tan_spot_f1': report.get('tan_spot', {}).get('f1-score', 0.0),
        'leaf_blight_f1': report.get('leaf_blight', {}).get('f1-score', 0.0),
        'full_report': report
    }
    
    return metrics

def evaluate_test_with_tta(model, test_loader, device, class_names, n_augments=7):
    """Évaluer sur test set avec TTA"""
    model.eval()
    y_true, y_pred = [], []
    
    print(f"\n🔄 Using Test-Time Augmentation ({n_augments} augmentations)...")
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            
            # TTA: moyenne de plusieurs prédictions
            predictions = []
            
            # Original
            outputs = model(inputs)
            predictions.append(F.softmax(outputs, dim=1))
            
            # Augmentations
            for _ in range(n_augments - 1):
                aug_inputs = inputs.clone()
                aug_type = np.random.randint(3)
                
                if aug_type == 0:  # Horizontal flip
                    aug_inputs = torch.flip(aug_inputs, [3])
                elif aug_type == 1:  # Vertical flip
                    aug_inputs = torch.flip(aug_inputs, [2])
                elif aug_type == 2:  # Brightness
                    brightness = 0.9 + 0.2 * np.random.rand()
                    aug_inputs = aug_inputs * brightness
                    aug_inputs = torch.clamp(aug_inputs, 0, 1)
                
                outputs = model(aug_inputs)
                predictions.append(F.softmax(outputs, dim=1))
            
            # Moyenne
            avg_pred = torch.stack(predictions).mean(0)
            _, preds = torch.max(avg_pred, 1)
            
            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())
    
    report = classification_report(
        y_true, y_pred, 
        target_names=class_names, 
        output_dict=True,
        zero_division=0
    )
    
    return report, y_true, y_pred

# =============================================================================
# Visualisation
# =============================================================================
def plot_fine_tuning_curves(history):
    """Afficher les courbes de fine-tuning"""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Loss
    axes[0, 0].plot(history['train_loss'], 'b-o', label='Train Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[0, 1].plot([a*100 for a in history['val_acc']], 'g-s', label='Val Accuracy')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy (%)')
    axes[0, 1].set_title('Validation Accuracy')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # F1 Scores
    axes[1, 0].plot([f*100 for f in history['tan_spot_f1']], 'r-o', label='tan_spot F1')
    axes[1, 0].plot([f*100 for f in history['leaf_blight_f1']], 'orange', marker='s', label='leaf_blight F1')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('F1-Score (%)')
    axes[1, 0].set_title('Difficult Classes F1-Scores')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Combined Difficult F1
    axes[1, 1].plot([f*100 for f in history['difficult_f1']], 'purple', marker='d', label='Average Difficult F1')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('F1-Score (%)')
    axes[1, 1].set_title('Average Difficult Classes F1')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'fine_tuning_curves.png'), dpi=300, bbox_inches='tight')
    print(f"✅ Fine-tuning curves saved to: {os.path.join(SAVE_DIR, 'fine_tuning_curves.png')}")
    plt.show()

# =============================================================================
# Main
# =============================================================================
if __name__ == '__main__':
    print("="*80)
    print("FINE-TUNING CONVNEXT MSF - Classes Difficiles")
    print("="*80)
    
    # Setup
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}\n")
    
    # 1. Charger données
    print("📦 Loading data...")
    train_loader, val_loader, test_loader, class_names, class_weights = get_dataloaders()
    print(f"✅ Data loaded: {len(train_loader.dataset)} train, "
          f"{len(val_loader.dataset)} val, {len(test_loader.dataset)} test")
    print(f"   Classes: {class_names}\n")
    
    # 2. Charger modèle pré-entraîné
    print("📦 Loading pre-trained model...")
    model = build_model(len(class_names)).to(device)
    
    # Essayer de charger le modèle
    if os.path.exists(PRETRAINED_MODEL_PATH):
        model.load_state_dict(torch.load(PRETRAINED_MODEL_PATH, map_location=device))
        print(f"✅ Model loaded from: {PRETRAINED_MODEL_PATH}")
    elif os.path.exists(LEGACY_MODEL_PATH):
        model.load_state_dict(torch.load(LEGACY_MODEL_PATH, map_location=device))
        print(f"✅ Model loaded from: {LEGACY_MODEL_PATH}")
    else:
        raise FileNotFoundError(
            f"❌ No pre-trained model found!\n"
            f"   Expected: {PRETRAINED_MODEL_PATH}\n"
            f"   Or: {LEGACY_MODEL_PATH}\n"
            f"   Please train the model first using train_convnext.ipynb"
        )
    
    # Évaluation avant fine-tuning
    print("\n" + "="*80)
    print("📊 EVALUATION BEFORE FINE-TUNING")
    print("="*80)
    before_metrics = evaluate_validation(model, val_loader, device, class_names)
    print(f"   Overall Accuracy: {before_metrics['overall_acc']:.4f}")
    print(f"   tan_spot F1: {before_metrics['tan_spot_f1']:.4f}")
    print(f"   leaf_blight F1: {before_metrics['leaf_blight_f1']:.4f}")
    
    # 3. Fine-tuning
    print("\n" + "="*80)
    print("🎯 STARTING FINE-TUNING")
    print("="*80)
    fine_tuned_model, history = fine_tune_difficult_classes(
        model, train_loader, val_loader, device,
        class_names, class_weights,
        difficult_classes=DIFFICULT_CLASSES,
        num_epochs=FINE_TUNE_EPOCHS,
        learning_rate=FINE_TUNE_LR
    )
    
    # 4. Évaluation après fine-tuning
    print("\n" + "="*80)
    print("📊 EVALUATION AFTER FINE-TUNING")
    print("="*80)
    after_metrics = evaluate_validation(fine_tuned_model, val_loader, device, class_names)
    print(f"   Overall Accuracy: {after_metrics['overall_acc']:.4f}")
    print(f"   tan_spot F1: {after_metrics['tan_spot_f1']:.4f}")
    print(f"   leaf_blight F1: {after_metrics['leaf_blight_f1']:.4f}")
    
    # Comparaison
    print("\n" + "="*80)
    print("📈 IMPROVEMENT COMPARISON")
    print("="*80)
    print(f"   Overall Accuracy: {before_metrics['overall_acc']:.4f} → {after_metrics['overall_acc']:.4f} "
          f"(+{(after_metrics['overall_acc'] - before_metrics['overall_acc'])*100:.2f}%)")
    print(f"   tan_spot F1: {before_metrics['tan_spot_f1']:.4f} → {after_metrics['tan_spot_f1']:.4f} "
          f"(+{(after_metrics['tan_spot_f1'] - before_metrics['tan_spot_f1'])*100:.2f}%)")
    print(f"   leaf_blight F1: {before_metrics['leaf_blight_f1']:.4f} → {after_metrics['leaf_blight_f1']:.4f} "
          f"(+{(after_metrics['leaf_blight_f1'] - before_metrics['leaf_blight_f1'])*100:.2f}%)")
    
    # 5. Évaluation finale sur test set avec TTA
    print("\n" + "="*80)
    print("🧪 FINAL TEST SET EVALUATION (WITH TTA)")
    print("="*80)
    test_report, y_true, y_pred = evaluate_test_with_tta(
        fine_tuned_model, test_loader, device, class_names, n_augments=7
    )
    
    print(f"\n✅ Final Test Results:")
    print(f"   Overall Accuracy: {test_report['accuracy']:.4f} ({test_report['accuracy']*100:.2f}%)")
    print(f"   tan_spot F1: {test_report.get('tan_spot', {}).get('f1-score', 0.0):.4f}")
    print(f"   leaf_blight F1: {test_report.get('leaf_blight', {}).get('f1-score', 0.0):.4f}")
    
    # Classification report complet
    print("\n" + "="*80)
    print("📋 FULL CLASSIFICATION REPORT")
    print("="*80)
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
    
    # 6. Visualisations
    print("\n📊 Generating visualizations...")
    plot_fine_tuning_curves(history)
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names, 
                cbar_kws={'label': 'Count'})
    plt.xlabel('Predicted', fontsize=12, fontweight='bold')
    plt.ylabel('True', fontsize=12, fontweight='bold')
    plt.title(f'Confusion Matrix - Fine-Tuned Model (Test Acc: {test_report["accuracy"]*100:.2f}%)', 
              fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'fine_tuned_confusion_matrix.png'), dpi=300, bbox_inches='tight')
    print(f"✅ Confusion matrix saved to: {os.path.join(SAVE_DIR, 'fine_tuned_confusion_matrix.png')}")
    plt.show()
    
    # 7. Sauvegarder résultats
    results = {
        'before_fine_tuning': before_metrics,
        'after_fine_tuning': after_metrics,
        'test_results': {
            'accuracy': test_report['accuracy'],
            'tan_spot_f1': test_report.get('tan_spot', {}).get('f1-score', 0.0),
            'leaf_blight_f1': test_report.get('leaf_blight', {}).get('f1-score', 0.0),
        },
        'improvements': {
            'overall_acc': (after_metrics['overall_acc'] - before_metrics['overall_acc']) * 100,
            'tan_spot_f1': (after_metrics['tan_spot_f1'] - before_metrics['tan_spot_f1']) * 100,
            'leaf_blight_f1': (after_metrics['leaf_blight_f1'] - before_metrics['leaf_blight_f1']) * 100,
        },
        'history': history
    }
    
    with open(os.path.join(SAVE_DIR, 'fine_tuning_results.json'), 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"\n✅ Results saved to: {os.path.join(SAVE_DIR, 'fine_tuning_results.json')}")
    print(f"✅ Fine-tuned model saved to: {FINE_TUNED_MODEL_PATH}")
    
    print("\n" + "="*80)
    print("✓ FINE-TUNING COMPLETE!")
    print("="*80)